# Fine-tune Qwen3-4B on `train.jsonl` → Export to Ollama

This notebook fine-tunes **Qwen3-4B** with LoRA on your `train.jsonl` (messages format) using **Unsloth**, then exports the result to **GGUF** so you can run it locally in **Ollama**.

**Runtime:** Colab Pro → `Runtime` → `Change runtime type` → **T4 GPU** (or A100 if available).

**Pipeline:**
1. Install Unsloth
2. Load Qwen3-4B in 4-bit + attach LoRA adapters
3. Load your `train.jsonl` and apply Qwen3's chat template
4. Train with TRL's `SFTTrainer`
5. Quick sanity-check inference
6. Export to GGUF (`q4_k_m`) and download
7. Run in Ollama on your local machine (instructions at the end)

> ⚠️ **Most common pitfall:** chat-template mismatch between training and inference. We use Unsloth's `get_chat_template("qwen3")` and a matching `Modelfile` template at the end to avoid this.

## 1. Install Unsloth

This takes a couple of minutes the first time. The `%%capture` keeps the output tidy.

In [1]:
%%capture
!pip install --upgrade --quiet unsloth
!pip install --upgrade --quiet --no-deps "trl<0.20" peft accelerate bitsandbytes
!pip install --upgrade --quiet transformers datasets

## 2. Get `train.jsonl` into the runtime

Pick **ONE** of the two cells below.

**Option A — Mount Google Drive** (recommended, file persists across sessions). Put `train.jsonl` somewhere in your Drive first.

**Option B — Upload directly** from your computer (gone when the runtime resets).

In [3]:
from google.colab import files
uploaded = files.upload()  # pick train.jsonl in the dialog
TRAIN_PATH = "/content/train.jsonl"
import shutil; shutil.move(list(uploaded.keys())[0], TRAIN_PATH)
print("Uploaded to:", TRAIN_PATH)

Saving train.jsonl to train.jsonl
Uploaded to: /content/train.jsonl


## 3. Load Qwen3-4B in 4-bit

Unsloth's pre-quantized 4-bit weights download faster and use ~3 GB VRAM. `max_seq_length=2048` is a safe default — bump it later if your conversations are longer.

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096   # covers 53/55 of your examples; only 2 long ones get truncated

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit   = True,
    dtype          = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.6.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 4. Attach LoRA adapters

LoRA lets us train a tiny fraction of the parameters and still get good results. `r=16` is a sensible default for most fine-tunes; raise to 32 or 64 if you have a lot of data and want more capacity.

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # saves VRAM, allows longer context
    random_state   = 3407,
)

[transformers] Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 5. Load `train.jsonl` and apply Qwen3's chat template

Your file has rows like:
```json
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

`standardize_sharegpt` normalizes that shape, then `apply_chat_template` wraps each turn in Qwen3's `<|im_start|>...<|im_end|>` markers — the exact same markers we'll use at inference.

In [6]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
print(f"Loaded {len(dataset)} examples; columns: {dataset.column_names}")

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 1129 examples; columns: ['messages']


Map:   0%|          | 0/1129 [00:00<?, ? examples/s]

Sanity check — make sure one formatted example looks right (should contain `<|im_start|>user`, `<|im_start|>assistant`, and `<|im_end|>` markers):

In [7]:
print(dataset[0]["text"])

<|im_start|>system
You are a clinical trial matching assistant. You receive a patient's condition list and must do two things:

1. Summarize the patient's main medical problems in 1-2 sentences.
2. Extract up to 32 key medical conditions ranked by clinical priority for searching clinical trials. Focus on specific, searchable medical terms.

Output ONLY a JSON object formatted as:
{"summary": "<summary>", "conditions": ["<condition1>", "<condition2>", ...]}<|im_end|>
<|im_start|>user
Here is the patient description:
Patient: male, DOB: 1974-06-03
Patient ID: ff2c3829-d4a3-a67d-19ce-c84d8eb76a9b

Current conditions (41):
- Gingival disease (disorder) (status: resolved, onset: 2024-12-02 10:38:29)
- Gingivitis (disorder) (status: resolved, onset: 2024-11-18 08:58:54)
- Polyp of colon (disorder) (status: resolved, onset: 2024-06-03 07:56:49)
- Acute viral pharyngitis (disorder) (status: resolved, onset: 2022-03-28 02:29:56)
- Fracture of bone (disorder) (status: resolved, onset: 2019-01-14

## 6. Train

Defaults below are tuned for a T4 with a few hundred to a few thousand examples. Pick **one** of:
- `num_train_epochs=1` for full passes (good for larger datasets)
- `max_steps=60` for a quick smoke test (good for tiny datasets / first run)

Monitor `train_loss` in the output — it should drop steadily. If it's flat or NaN, lower the learning rate to `1e-4`.

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = max_seq_length,
        per_device_train_batch_size = 1,    # was 2 — drop because seqs are longer now
        gradient_accumulation_steps = 8,    # effective batch still = 8
        warmup_steps                = 5,
        num_train_epochs            = 3,    # was 1 — bumped because only 55 examples
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 1,    # see loss every step on a small dataset
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:819: UserWarning: Padding-free training is enabled, but the attention implementation is not set to 'flash_attention_2'. Padding-free training flattens batches into a single sequence, and 'flash_attention_2' is the only known attention mechanism that reliably supports this. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation='flash_attention_2'` in the model configuration, or verify that your attention mechanism can handle flattened sequences.
  warnings.warn(


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1129 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=16):   0%|          | 0/1129 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1129 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,129 | Num Epochs = 3 | Total steps = 426
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.431674
2,1.394623
3,1.068148
4,1.059918
5,0.701731
6,1.086969
7,0.851745
8,0.671564
9,0.977885
10,0.839889


[transformers] Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-426/tokenizer_config.json.


## 7. Quick inference test

Make sure the model actually learned something before we spend 10 minutes exporting GGUF.

In [21]:
FastLanguageModel.for_inference(model)

messages = [{"role": "system", "content": "You are a clinical trial matching assistant. You receive a patient's condition list and must do two things:\n\n1. Summarize the patient's main medical problems in 1-2 sentences.\n2. Extract up to 32 key medical conditions ranked by clinical priority for searching clinical trials. Focus on specific, searchable medical terms.\n\nOutput ONLY a JSON object formatted as:\n{\"summary\": \"<summary>\", \"conditions\": [\"<condition1>\", \"<condition2>\", ...]}"}, {"role": "user", "content": "Here is the patient description:\nPatient: male, DOB: 1974-06-03\nPatient ID: ff2c3829-d4a3-a67d-19ce-c84d8eb76a9b\n\nCurrent conditions (41):\n- Gingival disease (disorder) (status: resolved, onset: 2024-12-02 10:38:29)\n- Gingivitis (disorder) (status: resolved, onset: 2024-11-18 08:58:54)\n- Polyp of colon (disorder) (status: resolved, onset: 2024-06-03 07:56:49)\n- Acute viral pharyngitis (disorder) (status: resolved, onset: 2022-03-28 02:29:56)\n- Fracture of bone (disorder) (status: resolved, onset: 2019-01-14 07:29:56)\n- Fracture of clavicle (disorder) (status: resolved, onset: 2019-01-14 07:29:56)\n- Essential hypertension (disorder) (status: active, onset: 2008-08-18 07:29:56)\n- Anemia (disorder) (status: active, onset: 1999-08-09 07:29:56)\n- Chronic sinusitis (disorder) (status: active, onset: 1997-07-17 14:29:56)\n- Loss of teeth (disorder) (status: active, onset: 1991-08-12 07:29:56)\n\nJSON output:"}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize               = True,
    add_generation_prompt  = True,
    return_tensors         = "pt",
).to("cuda")

outputs = model.generate(
    input_ids      = inputs,
    max_new_tokens = 128,
    temperature    = 0.7,
    top_p          = 0.8,
    top_k          = 20,
    do_sample      = True,
)
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[transformers] Both `max_new_tokens` (=128) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


<|im_start|>system
You are a clinical trial matching assistant. You receive a patient's condition list and must do two things:

1. Summarize the patient's main medical problems in 1-2 sentences.
2. Extract up to 32 key medical conditions ranked by clinical priority for searching clinical trials. Focus on specific, searchable medical terms.

Output ONLY a JSON object formatted as:
{"summary": "<summary>", "conditions": ["<condition1>", "<condition2>", ...]}<|im_end|>
<|im_start|>user
Here is the patient description:
Patient: male, DOB: 1974-06-03
Patient ID: ff2c3829-d4a3-a67d-19ce-c84d8eb76a9b

Current conditions (41):
- Gingival disease (disorder) (status: resolved, onset: 2024-12-02 10:38:29)
- Gingivitis (disorder) (status: resolved, onset: 2024-11-18 08:58:54)
- Polyp of colon (disorder) (status: resolved, onset: 2024-06-03 07:56:49)
- Acute viral pharyngitis (disorder) (status: resolved, onset: 2022-03-28 02:29:56)
- Fracture of bone (disorder) (status: resolved, onset: 2019-01-14

## 8. Export to GGUF for Ollama

This step builds `llama.cpp` from source on first run (5–10 minutes), then quantizes the merged model. `q4_k_m` is the standard sweet spot for Ollama — small enough to run on CPU/laptop GPUs, good quality.

Other choices if you want them:
- `q8_0` — bigger (~4.3 GB), higher quality
- `f16` — full precision (~8 GB)

In [16]:
# Saves the merged base + LoRA as a single GGUF file in ./qwen3-4b-finetuned/
model.save_pretrained_gguf(
    "qwen3-4b-finetuned",
    tokenizer,
    quantization_method = "q4_k_m",
)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

[transformers] Unsloth: Restored added_tokens_decoder metadata in qwen3-4b-finetuned/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:30<00:00, 15.01s/it]


Unsloth: Merge process complete. Saved to `/content/qwen3-4b-finetuned`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen3-4b-finetuned_gguf/qwen3-4b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen3-4b-finetuned_gguf/qwen3-4b.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model qwen3-4b-finetuned_gguf/qwen3-4b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to qwen3-4b-finetuned_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f qwen3-4b-finetuned_gguf/Modelfile


{'save_directory': 'qwen3-4b-finetuned',
 'gguf_directory': 'qwen3-4b-finetuned_gguf',
 'gguf_files': ['qwen3-4b-finetuned_gguf/qwen3-4b.Q4_K_M.gguf'],
 'modelfile_location': 'qwen3-4b-finetuned_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 9. Get the GGUF off Colab

Pick whichever fits your setup. Drive is fastest if you mounted it; direct download works for files up to a few GB.

In [17]:
# Find the produced GGUF file
import glob, os
gguf_files = glob.glob("qwen3-4b-finetuned_gguf/*.gguf")
print("GGUFs:", gguf_files)
GGUF_PATH = gguf_files[0]
print("Size:", round(os.path.getsize(GGUF_PATH) / 1e9, 2), "GB")

GGUFs: ['qwen3-4b-finetuned_gguf/qwen3-4b.Q4_K_M.gguf']
Size: 2.5 GB


In [20]:
# Find the produced GGUF file (copied from the previous cell to ensure GGUF_PATH is defined)
import glob, os

# Correct the directory name based on the actual file system state
gguf_files = glob.glob("qwen3-4b-finetuned_gguf/*.gguf")

# If no GGUF file is found with the new path, try the original path as a fallback
if not gguf_files:
    gguf_files = glob.glob("qwen3-4b-finetuned/*.gguf")

if not gguf_files:
    raise FileNotFoundError("No GGUF file found in 'qwen3-4b-finetuned_gguf/' or 'qwen3-4b-finetuned/' directories. Please ensure the previous export step was successful.")

GGUF_PATH = gguf_files[0]
print("GGUF found:", GGUF_PATH)
print("Size:", round(os.path.getsize(GGUF_PATH) / 1e9, 2), "GB")

# OPTION A: copy to Drive
import shutil
# Make sure to mount Google Drive first if you haven't already
from google.colab import drive
drive.mount('/content/drive')
shutil.copy(GGUF_PATH, "/content/drive/MyDrive/qwen3-4b-finetuned.gguf")
print("Copied to Drive.")

# OPTION B: direct download to your computer
# from google.colab import files
# files.download(GGUF_PATH)

GGUF found: qwen3-4b-finetuned_gguf/qwen3-4b.Q4_K_M.gguf
Size: 2.5 GB
Mounted at /content/drive
Copied to Drive.


## 10. Run it in Ollama (on your local machine)

Everything below runs on **your computer**, not in Colab.

### a) Install Ollama
- macOS / Windows: download from <https://ollama.com/download>
- Linux: `curl -fsSL https://ollama.com/install.sh | sh`

### b) Put the GGUF and a `Modelfile` in the same folder

Save the file you downloaded as `qwen3-4b-finetuned.gguf`. Next to it, create a plain text file named `Modelfile` (no extension) with this content:

```
FROM ./qwen3-4b-finetuned.gguf

TEMPLATE """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER num_ctx 4096
```

> The `TEMPLATE` block is the **same** ChatML format Unsloth used during training — that's what prevents gibberish output.

### c) Build and run

```bash
cd /path/to/folder/with/Modelfile
ollama create qwen-tuned -f Modelfile
ollama run qwen-tuned
```

Then chat away. To use it from Python or any OpenAI-compatible client, point at `http://localhost:11434`.

### Troubleshooting

| Symptom | Fix |
|---|---|
| Endless output / repeats | Bump `num_ctx` to `32768`; check stop tokens are present |
| Gibberish | Template mismatch — copy the `TEMPLATE` block exactly as above |
| Out of memory | Use a smaller quant (re-export with `q4_k_s` or `q3_k_m`) |
| Model behaves like base Qwen | LoRA didn't merge — re-run cell 8, confirm the GGUF is fresh |
